# Lezione 21 — Metriche di retrieval: Recall@K, Precision@K, MRR

**Come si usa questo notebook.** Come sempre. Prerequisiti: Lezione 18
(similarita' coseno) e Lezione 20 (valutare senza confondere accuratezza
con la metrica giusta). Chiude la Fase 3 (testo ed embedding) del corso.

Finora abbiamo *classificato* (Lezioni 15-17: predire il `type`) e
*raggruppato* (Lezione 20: K-Means). Oggi il compito e' un terzo, diverso
da entrambi: la **ricerca** (*retrieval*) — data una memoria, restituire
un elenco **ordinato** delle memorie piu' rilevanti. Un elenco ordinato
serve metriche diverse da un'accuratezza di classificazione.

**Cosa saprai fare alla fine:** definire e calcolare Precision@K,
Recall@K e Mean Reciprocal Rank, e spiegare quando ciascuna e' la
metrica giusta da guardare per un sistema di ricerca.

## Parte 1 — Teoria: valutare un elenco ordinato

Un sistema di retrieval, data una **query** (qui: una memoria), restituisce
un **ranking**: tutte le altre memorie ordinate per similarita' (coseno,
Lezione 18) decrescente. Il primo problema pratico e' definire cosa vuol
dire "corretto": qui, in assenza di giudizi di rilevanza fatti a mano,
usiamo una **proxy dichiarata esplicitamente**: una memoria e' "rilevante"
per la query se ha lo **stesso `type`**. E' un'approssimazione, non una
verita' assoluta (due memorie *episodic* possono essere semanticamente
lontane, due memorie di type diverso potrebbero essere correlate) — ma e'
un proxy misurabile con le etichette che abbiamo, dichiarato come tale
invece di spacciarlo per una verita' di rilevanza definitiva.

Tre metriche, tre domande diverse sullo stesso ranking:

- **Precision@K** — dei primi `K` risultati restituiti, quanti sono
  rilevanti? `rilevanti_in_top_K / K`. Risponde a "quanto e' pulito quello
  che mostro subito all'utente" — importante quando l'utente guarda solo
  i primi risultati (es. 3 memorie suggerite in un'interfaccia).
- **Recall@K** — di tutti gli elementi rilevanti che esistono, quanti sono
  finiti nei primi `K`? `rilevanti_in_top_K / rilevanti_totali`. Risponde
  a "quanto del materiale utile sto trovando" — importante quando serve
  *tutto* cio' che e' rilevante, non solo un assaggio. Con `K` piccolo e
  molti elementi rilevanti (una classe grande), Recall@K e' strutturalmente
  basso anche con un ranking ottimo: e' aritmetica, non un difetto del
  motore di ricerca.
- **Mean Reciprocal Rank (MRR)** — per ogni query, `1 / rank` della
  **prima** posizione rilevante trovata (rank `1` se il primo risultato e'
  gia' rilevante, `1/2` se serve arrivare al secondo, ecc.), poi la media
  su tutte le query. Risponde a "quanto in fretta trovo *almeno un*
  risultato buono" — importante quando basta un solo risultato giusto
  (es. "trova la memoria piu' simile per rispondere subito").

Nessuna delle tre e' "la" metrica giusta in assoluto: la scelta dipende da
cosa deve fare il sistema con i risultati.

In [1]:
import numpy as np

# Ranking giocattolo: 6 elementi, rilevanza nota (1 = rilevante, 0 = no),
# gia' ordinati per similarita' decrescente (come restituirebbe un motore di ricerca).
rilevanza_ranking = np.array([1, 0, 1, 0, 0, 1])  # posizioni 1..6

K = 3
rilevanti_in_topk = rilevanza_ranking[:K].sum()
rilevanti_totali = rilevanza_ranking.sum()

precision_a_k = rilevanti_in_topk / K
recall_a_k = rilevanti_in_topk / rilevanti_totali
print(f'Precision@{K}: {precision_a_k:.3f}  ({rilevanti_in_topk} rilevanti su {K} mostrati)')
print(f'Recall@{K}: {recall_a_k:.3f}  ({rilevanti_in_topk} rilevanti su {rilevanti_totali} totali)')

rank_primo_rilevante = np.argmax(rilevanza_ranking) + 1  # argmax trova il primo 1
print(f'Reciprocal Rank: {1 / rank_primo_rilevante:.3f}  (primo rilevante in posizione {rank_primo_rilevante})')

Precision@3: 0.667  (2 rilevanti su 3 mostrati)
Recall@3: 0.667  (2 rilevanti su 3 totali)
Reciprocal Rank: 1.000  (primo rilevante in posizione 1)


Leggi l'output: nei primi 3 risultati (`[1, 0, 1]`) 2 su 3 sono rilevanti
(`Precision@3 = 0.667`); su un totale di 3 elementi rilevanti nell'intero
ranking, 2 sono finiti nei primi 3 (`Recall@3 = 0.667` — coincidenza
numerica di questo esempio, non una regola generale: guarda cosa succede
cambiando `K` nell'esercizio). Il primo elemento rilevante e' gia' in
posizione 1, quindi il Reciprocal Rank e' `1.0`, il massimo possibile.

## Parte 2 — Esercizio guidato

Il tuo compito: ricalcola Precision@K e Recall@K sullo stesso
`rilevanza_ranking` con `K = 6` (l'intero elenco). Cosa succede a
Recall@K quando `K` copre tutti gli elementi? E a Precision@K?

In [2]:
# Scrivi qui: K = 6, ricalcola precision_a_k e recall_a_k su rilevanza_ranking.

pass

### Soluzione spiegata

Con `K` pari alla lunghezza dell'intero elenco, **tutti** gli elementi
rilevanti sono necessariamente inclusi nei "primi K": Recall@K diventa
`1.0` per costruzione, qualunque sia la qualita' del ranking (persino un
ranking casuale avrebbe Recall@K = 1.0 con `K` = tutto). Precision@K
invece diventa semplicemente la frazione di elementi rilevanti
sull'intero pool (`3/6 = 0.5` qui) — non dice piu' niente sull'**ordine**
del ranking, solo sulla composizione dell'insieme. Questo e' il motivo per
cui Recall@K e Precision@K si guardano sempre **insieme**, con un `K`
realistico (non l'intero elenco): da soli, ai due estremi di `K`,
diventano poco informativi.

In [3]:
K = 6
rilevanti_in_topk = rilevanza_ranking[:K].sum()
precision_a_k = rilevanti_in_topk / K
recall_a_k = rilevanti_in_topk / rilevanti_totali
print(f'Precision@{K}: {precision_a_k:.3f}')
print(f'Recall@{K}: {recall_a_k:.3f}')
assert recall_a_k == 1.0

Precision@6: 0.500
Recall@6: 1.000


## Parte 3 — Il progetto: Memory AI Lab, passo 21 — misurare la ricerca

Ultimo passo del modulo Fase 3: ogni memoria di validation come query, le
altre come pool di ricerca, ranking per similarita' coseno
(incorporatore delle Lezioni 17-20), rilevanza = stesso `type`. Calcoliamo
Precision@3, Recall@3 e MRR su tutte le query — escludendo le query per
cui **non esiste nessun altro elemento rilevante nel pool** (Recall e
Precision non sono definite se il denominatore o il numero di elementi
rilevanti disponibili e' zero: succede con le classi rappresentate da una
sola memoria nel validation set, come visto nella Lezione 20).

In [4]:
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import pandas as pd
import keras
from keras.layers import TextVectorization
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

keras.utils.set_random_seed(42)

processed = Path('..') / 'datasets' / 'processed'
memorie = {n: pd.read_csv(processed / f'memory_{n}.csv') for n in ['train', 'val']}
classi = ['episodic', 'preference', 'semantic', 'unknown']
mappa = {c: i for i, c in enumerate(classi)}
testi = {n: f['text'].astype(str).to_numpy() for n, f in memorie.items()}
target = {n: f['type'].map(mappa).to_numpy() for n, f in memorie.items()}

LUNGHEZZA_SEQUENZA = 24
vettorizzatore = TextVectorization(max_tokens=300, output_mode='int',
                                   output_sequence_length=LUNGHEZZA_SEQUENZA)
vettorizzatore.adapt(testi['train'])
X_seq = {n: vettorizzatore(t).numpy() for n, t in testi.items()}

ingresso = keras.Input(shape=(LUNGHEZZA_SEQUENZA,))
parole = keras.layers.Embedding(input_dim=300, output_dim=16, mask_zero=True)(ingresso)
vettore_frase = keras.layers.GlobalAveragePooling1D(name='sentence_embedding')(parole)
nascosto = keras.layers.Dense(32, activation='relu')(vettore_frase)
nascosto = keras.layers.Dropout(0.3)(nascosto)
uscita = keras.layers.Dense(4, activation='softmax')(nascosto)

classificatore = keras.Model(ingresso, uscita)
classificatore.compile(loss='sparse_categorical_crossentropy', optimizer='adam',
                       metrics=['accuracy'])
classificatore.fit(X_seq['train'], target['train'], epochs=300,
                   validation_data=(X_seq['val'], target['val']),
                   callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=20,
                                                            restore_best_weights=True)],
                   verbose=0)

incorporatore = keras.Model(classificatore.input,
                            classificatore.get_layer('sentence_embedding').output)
vettori_val = incorporatore(X_seq['val']).numpy()
tipi_val = memorie['val']['type'].to_numpy()
n = len(tipi_val)

matrice_similarita = cosine_similarity(vettori_val)
np.fill_diagonal(matrice_similarita, -np.inf)  # una memoria non e' mai risultato di se' stessa

2026-07-16 21:27:55.094204: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
K = 3
precisioni, recall_lista, reciprocal_ranks = [], [], []
query_escluse = 0

for i in range(n):
    rilevanti_totali = np.sum(tipi_val == tipi_val[i]) - 1  # esclude la query stessa
    if rilevanti_totali == 0:
        query_escluse += 1
        continue  # nessun altro elemento rilevante nel pool: metriche non definite

    ranking = np.argsort(matrice_similarita[i])[::-1]
    topk = ranking[:K]
    rilevanti_in_topk = np.sum(tipi_val[topk] == tipi_val[i])

    precisioni.append(rilevanti_in_topk / K)
    recall_lista.append(rilevanti_in_topk / rilevanti_totali)

    for rank, idx in enumerate(ranking, start=1):
        if tipi_val[idx] == tipi_val[i]:
            reciprocal_ranks.append(1 / rank)
            break

print(f'query valutate: {len(precisioni)} / {n} ({query_escluse} escluse: nessun altro elemento rilevante)')
print(f'Precision@{K}: {np.mean(precisioni):.3f}')
print(f'Recall@{K}:    {np.mean(recall_lista):.3f}')
print(f'MRR:           {np.mean(reciprocal_ranks):.3f}')

query valutate: 18 / 20 (2 escluse: nessun altro elemento rilevante)
Precision@3: 0.907
Recall@3:    0.319
MRR:           1.000


Guarda i tre numeri insieme, non isolatamente. Nell'esecuzione di
riferimento: **Precision@3 alta** (i primi 3 risultati sono quasi sempre
dello stesso type — coerente con la separazione vista nelle Lezioni 18 e
20), **Recall@3 bassa** (la classe `episodic`, con 13 memorie nel
validation set, ha molti piu' di 3 elementi rilevanti: nessun `K = 3`
potrebbe mai coprirli tutti — e' aritmetica, non un difetto
dell'embedding), **MRR massimo** (per ogni query valutata, il primissimo
risultato e' gia' rilevante). I tre numeri raccontano storie diverse dello
**stesso** ranking: nessuno da solo basterebbe a descriverlo onestamente.

## Cosa hai imparato — e la Fase 3 e' completa

- Il **retrieval** valuta un **ranking**, non una singola predizione:
  serve una nozione dichiarata di "rilevante" (qui: stesso `type`, un
  proxy imperfetto ma misurabile).
- **Precision@K** (pulizia dei primi K risultati), **Recall@K**
  (copertura di tutto cio' che e' rilevante) e **MRR** (velocita' nel
  trovare almeno un risultato buono) rispondono a domande diverse:
  vanno lette insieme, non isolatamente.
- Recall@K e' strutturalmente basso quando gli elementi rilevanti sono
  molti piu' di `K`: non e' un segnale di scarsa qualita' del ranking di
  per se'.
- **Chiudi qui la Fase 3.** Dal testo grezzo (Lezione 15) sei arrivato a
  un sistema che rappresenta le memorie come vettori (Lezioni 16-17), le
  confronta (Lezione 18), le visualizza (Lezione 19), le raggruppa senza
  etichette (Lezione 20) e valuta onestamente la qualita' della ricerca
  (oggi) — gli ingredienti di base di un sistema di memoria semantica.

## Quiz

1. Perche' Recall@K puo' essere strutturalmente basso anche con un
   ranking di ottima qualita', se la classe rilevante e' molto numerosa?
2. Un sistema ha MRR = 1.0 ma Recall@10 = 0.2. Cosa ti dice questo sul
   comportamento del ranking, in positivo e in negativo?
3. Perche' "stesso `type`" e' definito esplicitamente come un **proxy**
   di rilevanza, e non semplicemente "la rilevanza"?

<details>
<summary><b>Apri le risposte</b></summary>

1. Perche' Recall@K = `rilevanti_in_top_K / rilevanti_totali`: se
   `rilevanti_totali` e' molto maggiore di `K`, anche includendo *solo*
   elementi rilevanti nei primi `K` (il caso migliore possibile) il
   rapporto non puo' superare `K / rilevanti_totali` — un tetto
   aritmetico indipendente dalla qualita' del ranking.
2. In positivo: per ogni query, il primissimo risultato e' quasi sempre
   rilevante (il sistema trova velocemente *un* buon risultato). In
   negativo: tra i primi 10, solo il 20% degli elementi rilevanti totali
   compare — il sistema non e' bravo a coprire *tutto* cio' che e'
   rilevante, solo a trovarne uno rapidamente. Sistemi diversi (un
   "trova la risposta" contro un "mostrami tutto il materiale correlato")
   richiederebbero letture opposte dello stesso risultato.
3. Perche' due memorie con lo stesso `type` non sono necessariamente
   semanticamente vicine (es. due `episodic` su eventi completamente
   diversi), e due memorie di type diverso potrebbero comunque essere
   correlate: usare il `type` e' una scelta pratica per avere una
   rilevanza misurabile con le etichette disponibili, non un'affermazione
   che quella sia l'unica nozione corretta di rilevanza.
</details>

## Fonti

- scikit-learn, `sklearn.metrics.pairwise.cosine_similarity` (gia' usata
  nella Lezione 18, riusata qui per il ranking):
  https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html
- Wikipedia, *Mean reciprocal rank* (definizione di riferimento,
  ampiamente citata in letteratura di information retrieval):
  https://en.wikipedia.org/wiki/Mean_reciprocal_rank

## Prossima modulo

La Fase 3 (testo ed embedding) e' chiusa. Il modulo successivo affronta
la **rappresentazione delle memorie**: schema, importanza, relazioni tra
entita' ed eventi, memoria a grafo — costruendo sopra gli embedding di
oggi invece che sopra il solo testo grezzo.